# 🛍️ Proyek Analisis Data: E-Commerce Public Dataset (Olist)
**Domain:** E-Commerce Marketplace Intelligence, Category Performance, & Customer Satisfaction  
**Posisi:** Junior Data Analyst Portfolio  
**Latar Belakang Analis:** Matematika (Fokus pada Ketelitian Data, Statistik Deskriptif, dan Visual Storytelling)

---

## 📌 Latar Belakang & Pernyataan Masalah
E-commerce marketplace modern menghadapi tantangan ganda: memaksimalkan pertumbuhan pendapatan sekaligus menjaga kualitas layanan pelanggan. Proyek ini menganalisis dataset publik transaksi e-commerce Brasil (Olist) yang mencakup lebih dari 100.000 transaksi dari tahun 2016 hingga 2018.

Tujuan utama analisis ini adalah memberikan wawasan kuantitatif berbasis data bagi pengambil keputusan komersial dan operasional, dengan memfokuskan pada tiga pilar analitis:
1. **Dinamika Penjualan & Pola Pertumbuhan Bulanan (MoM):** Memahami evolusi pendapatan bersih, volume pesanan, dan mendeteksi anomali musiman (seperti Black Friday).
2. **Distribusi Portofolio Kategori Produk (Prinsip Pareto 80/20):** Memetakan kategori produk penggerak utama (*core revenue drivers*) dan mengidentifikasi variasi Average Order Value (AOV) dari total 72 kategori produk aktif.
3. **Pengaruh Logistik terhadap Kepuasan Pelanggan:** Menguji secara empiris korelasi antara durasi pengiriman riil serta deviasi estimasi (keterlambatan berbasis kalender) terhadap skor ulasan (rating 1-5 bintang).

---

## 🎯 Menentukan 3 Pertanyaan Bisnis SMART
1. **Pertanyaan 1 (Dinamika Pendapatan Bulanan & MoM):**  
   *Bagaimana dinamika pendapatan realisasi bulanan (Net Realized Revenue) dan laju pertumbuhannya (Month-over-Month Growth %) selama periode Januari 2017 hingga Agustus 2018, serta kapan titik puncak penjualan (peak season) terjadi?*
2. **Pertanyaan 2 (Analisis Pareto 80/20 Kategori Produk):**  
   *Berapa banyak dan kategori produk mana saja yang menyumbang 80% pendapatan kumulatif (Prinsip Pareto 80/20), serta bagaimana nilai rata-rata pesanan (Average Order Value / AOV) pada kategori-kategori utama tersebut?*
3. **Pertanyaan 3 (Pengaruh Durasi Pengiriman terhadap Kepuasan Pelanggan):**  
   *Bagaimana hubungan korelasi antara durasi pengiriman riil serta keterlambatan pengiriman terhadap skor ulasan pelanggan (review score 1–5), dan seberapa signifikan penurunan skor kepuasan pada pesanan yang terlambat dibandingkan pesanan tepat waktu?*


## 🧮 1. Import Library & Konfigurasi Lingkungan


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Konfigurasi visualisasi
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['figure.dpi'] = 100

print("Library berhasil dimuat.")


## 🧹 2. Data Wrangling

Tahap data wrangling terdiri atas tiga tahapan sistematis:
1. **Gathering Data:** Memuat 5 tabel relasional mentah dari direktori `data/`.
2. **Assessing Data:** Memeriksa tipe data, missing values, dan konsistensi tanggal.
3. **Cleaning Data & Feature Engineering:** Menyaring data pesanan delivered pada periode stabil (Jan 2017 s.d. Agu 2018), menerapkan normalisasi tanggal kalender untuk perhitungan keterlambatan SLA, serta mengekspor dataset bersih `clean_orders.csv`.


### 2.1 Gathering Data


In [ ]:
# Membaca 5 file dataset relasional (fleksibel: data/raw atau data/)
raw_dir = "data/raw" if os.path.exists("data/raw") else "data"
orders_df = pd.read_csv(os.path.join(raw_dir, "orders_dataset.csv"))
items_df = pd.read_csv(os.path.join(raw_dir, "order_items_dataset.csv"))
products_df = pd.read_csv(os.path.join(raw_dir, "products_dataset.csv"))
translation_df = pd.read_csv(os.path.join(raw_dir, "product_category_name_translation.csv"))
reviews_df = pd.read_csv(os.path.join(raw_dir, "order_reviews_dataset.csv"))

print(f"Jumlah baris orders_df: {len(orders_df):,}")
print(f"Jumlah baris items_df: {len(items_df):,}")
print(f"Jumlah baris products_df: {len(products_df):,}")
print(f"Jumlah baris reviews_df: {len(reviews_df):,}")


### 2.2 Assessing Data
Memeriksa struktur kolom, tipe data, dan missing values pada tabel utama.


In [ ]:
# Cek info dan missing value tabel orders
print("--- Missing Values pada orders_df ---")
print(orders_df.isna().sum()[orders_df.isna().sum() > 0])

# Cek distribusi status pesanan
print("\n--- Distribusi Status Pesanan ---")
print(orders_df['order_status'].value_counts())


### 2.3 Cleaning Data & Feature Engineering
Langkah-langkah pembersihan:
1. Mengonversi kolom timestamp ke tipe `datetime64`.
2. Menyaring pesanan: status `'delivered'`, memiliki tanggal pengiriman valid, dan rentang pembelian **1 Januari 2017 hingga 31 Agustus 2018 (20 bulan stabil)**.
3. Menggabungkan kategori produk dengan nama bahasa Inggris (mengisi nilai hilang dengan `'others'`).
4. **Presisi Keterlambatan Kalender:** Menghitung durasi pengiriman riil (`actual_delivery_days`), durasi estimasi (`estimated_delivery_days`), dan keterlambatan (`delay_days`) dengan menormalisasikan tanggal ke tingkat hari (`.dt.floor('D')`) agar pesanan yang tiba di siang hari pada tanggal estimasi tidak salah dicap terlambat.
5. Mendeduplikasi ulasan (rata-rata skor per pesanan) untuk mencegah penggandaan baris.
6. Menggabungkan semua informasi ke dalam `clean_orders.csv`.


In [ ]:
# 1. Konversi timestamp
date_cols = ['order_purchase_timestamp', 'order_approved_at', 
             'order_delivered_carrier_date', 'order_delivered_customer_date', 
             'order_estimated_delivery_date']
for col in date_cols:
    orders_df[col] = pd.to_datetime(orders_df[col])

# 2. Filter pesanan delivered pada periode stabil 2017-01 s.d. 2018-08
mask_delivered = orders_df['order_status'] == 'delivered'
mask_dates = (orders_df['order_purchase_timestamp'] >= '2017-01-01') & (orders_df['order_purchase_timestamp'] <= '2018-08-31 23:59:59')
mask_valid_delivery = orders_df['order_delivered_customer_date'].notna()

valid_orders = orders_df[mask_delivered & mask_dates & mask_valid_delivery].copy()

# 3. Feature engineering waktu
valid_orders['order_year_month'] = valid_orders['order_purchase_timestamp'].dt.to_period('M').astype(str)
valid_orders['actual_delivery_days'] = ((valid_orders['order_delivered_customer_date'] - valid_orders['order_purchase_timestamp']).dt.total_seconds() / 86400).round(2)
valid_orders['estimated_delivery_days'] = ((valid_orders['order_estimated_delivery_date'] - valid_orders['order_purchase_timestamp']).dt.total_seconds() / 86400).round(2)

# Normalisasi tanggal kalender (.dt.floor('D')) untuk delay_days dan is_delayed
date_delivered_calendar = valid_orders['order_delivered_customer_date'].dt.floor('D')
date_estimated_calendar = valid_orders['order_estimated_delivery_date'].dt.floor('D')
valid_orders['delay_days'] = ((date_delivered_calendar - date_estimated_calendar).dt.total_seconds() / 86400).round(2)
valid_orders['is_delayed'] = valid_orders['delay_days'] > 0

# 4. Penerjemahan kategori produk
prod_trans = products_df.merge(translation_df, on='product_category_name', how='left')
prod_trans['product_category_name_english'] = prod_trans['product_category_name_english'].fillna('others')

# 5. Agregasi review score per order (deduplikasi ulasan)
rev_agg = reviews_df.groupby('order_id', as_index=False)['review_score'].mean()
rev_agg['review_score'] = rev_agg['review_score'].round(2)

# 6. Penggabungan items, products, orders, dan reviews
items_merged = items_df[['order_id', 'order_item_id', 'product_id', 'price', 'freight_value']].merge(
    prod_trans[['product_id', 'product_category_name_english']], on='product_id', how='left'
)
items_merged['product_category_name_english'] = items_merged['product_category_name_english'].fillna('others')
items_merged['total_order_value'] = (items_merged['price'] + items_merged['freight_value']).round(2)

clean_df = items_merged.merge(
    valid_orders[['order_id', 'order_status', 'order_purchase_timestamp', 
                  'order_delivered_customer_date', 'order_estimated_delivery_date', 
                  'order_year_month', 'actual_delivery_days', 'estimated_delivery_days',
                  'delay_days', 'is_delayed']],
    on='order_id', how='inner'
)
clean_df = clean_df.merge(rev_agg, on='order_id', how='left')

# Susun ulang kolom
final_columns = [
    'order_id', 'order_status', 'order_purchase_timestamp',
    'order_delivered_customer_date', 'order_estimated_delivery_date',
    'order_year_month', 'product_id', 'product_category_name_english',
    'price', 'freight_value', 'total_order_value',
    'actual_delivery_days', 'estimated_delivery_days',
    'delay_days', 'is_delayed', 'review_score'
]
clean_df = clean_df[final_columns]

# Ekspor ke data/clean_orders.csv
clean_df.to_csv('data/clean_orders.csv', index=False)
print(f"Data bersih berhasil diekspor ke data/clean_orders.csv dengan {len(clean_df):,} baris transaksi.")


## 📊 3. Exploratory Data Analysis (EDA) & Statistik Terapan

Pada bagian ini, kita menjawab secara komprehensif 3 Pertanyaan Bisnis SMART yang telah dirumuskan:
- **Q1:** Dinamika Pendapatan Bulanan & Laju Pertumbuhan MoM.
- **Q2:** Analisis Distribusi Kumulatif Pareto 80/20 Portofolio Kategori Produk & AOV.
- **Q3:** Pengaruh Durasi & Keterlambatan Pengiriman terhadap Kepuasan Pelanggan.


### 3.1 Pertanyaan 1: Dinamika Pendapatan Bulanan & Laju Pertumbuhan (MoM)
> *"Bagaimana dinamika pendapatan realisasi bulanan (Net Realized Revenue) dan laju pertumbuhannya (Month-over-Month Growth %) selama periode Januari 2017 hingga Agustus 2018, serta kapan titik puncak penjualan (peak season) terjadi?"*


In [ ]:
# Agregasi penjualan bulanan
monthly_sales = clean_df.groupby('order_year_month').agg(
    total_revenue=('total_order_value', 'sum'),
    total_orders=('order_id', 'nunique'),
    total_items=('product_id', 'count')
).reset_index()

# Hitung MoM Growth Rate % dan AOV
monthly_sales['mom_growth_pct'] = (monthly_sales['total_revenue'].pct_change() * 100).round(2)
monthly_sales['aov'] = (monthly_sales['total_revenue'] / monthly_sales['total_orders']).round(2)

print("--- Ringkasan Kinerja Penjualan Bulanan (2017 - 2018) ---")
print(monthly_sales[['order_year_month', 'total_revenue', 'total_orders', 'mom_growth_pct', 'aov']])


#### Visualisasi Tren Penjualan & MoM Growth


In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Plot 1: Tren Pendapatan dan Total Order
color_rev = '#1E3A8A'
color_ord = '#10B981'

ax1.plot(monthly_sales['order_year_month'], monthly_sales['total_revenue'], marker='o', linewidth=2.5, color=color_rev, label='Net Revenue (BRL)')
ax1.set_ylabel('Total Revenue (BRL)', color=color_rev, fontsize=12, weight='bold')
ax1.tick_params(axis='y', labelcolor=color_rev)
ax1.set_title('Tren Pendapatan Bersih & Jumlah Pesanan Bulanan (Jan 2017 - Agu 2018)', fontsize=14, weight='bold', pad=15)

ax1_twin = ax1.twinx()
ax1_twin.plot(monthly_sales['order_year_month'], monthly_sales['total_orders'], marker='s', linewidth=2, linestyle='--', color=color_ord, label='Total Orders')
ax1_twin.set_ylabel('Total Pesanan (Orders)', color=color_ord, fontsize=12, weight='bold')
ax1_twin.tick_params(axis='y', labelcolor=color_ord)
ax1_twin.grid(False)

# Sorotan Black Friday Nov 2017
bf_rev = monthly_sales.loc[monthly_sales['order_year_month'] == '2017-11', 'total_revenue'].values[0]
ax1.annotate('Peak: Black Friday Nov 2017\n(BRL 1.15M | +53.5% MoM)', 
             xy=('2017-11', bf_rev), xytext=('2017-07', bf_rev + 150000),
             arrowprops=dict(facecolor='#EF4444', shrink=0.08, width=2, headwidth=8),
             fontsize=10, weight='bold', color='#B91C1C',
             bbox=dict(boxstyle='round,pad=0.4', facecolor='#FEE2E2', edgecolor='#EF4444'))

# Plot 2: MoM Growth %
colors_bar = ['#10B981' if x >= 0 else '#EF4444' for x in monthly_sales['mom_growth_pct'].fillna(0)]
ax2.bar(monthly_sales['order_year_month'], monthly_sales['mom_growth_pct'].fillna(0), color=colors_bar, width=0.6)
ax2.axhline(0, color='black', linestyle='-', linewidth=0.8)
ax2.set_ylabel('MoM Growth Rate (%)', fontsize=12, weight='bold')
ax2.set_xlabel('Periode Transaksi (Tahun-Bulan)', fontsize=12, weight='bold')
ax2.set_title('Laju Pertumbuhan Bulanan (Month-over-Month Growth %)', fontsize=13, weight='bold')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


💡 **Insight Bisnis & Statistik (Pertanyaan 1):**
1. **Pola Pertumbuhan:** Pendapatan bersih bulanan tumbuh secara konsisten dari **BRL 127.482** (Januari 2017) menjadi stabil di kisaran **BRL 1,0 Juta – 1,1 Juta** per bulan pada pertengahan 2018 (kenaikan volume >8 kali lipat).
2. **Puncak Musiman (Black Friday Nov 2017):** Terjadi lonjakan rekor penjualan pada November 2017 mencapai **BRL 1.153.229** dengan pertumbuhan bulanan sebesar **+53,54% MoM**. Lonjakan ini disusul oleh kontraksi pada Desember 2017 (-26,89%) sebagai efek normalisasi paska-promosi.
3. **Stabilitas Nilai Rata-rata Pesanan (AOV):** Meskipun volume pesanan meningkat pesat, nilai AOV bulanan stabil di kisaran **BRL 150 – BRL 170**, mengindikasikan bahwa pertumbuhan pendapatan didorong oleh akuisisi volume pesanan baru secara luas, bukan oleh kenaikan harga rata-rata.


### 3.2 Pertanyaan 2: Analisis Pareto 80/20 Portofolio Kategori Produk
> *"Berapa banyak dan kategori produk mana saja yang menyumbang 80% pendapatan kumulatif (Prinsip Pareto 80/20), serta bagaimana nilai rata-rata pesanan (Average Order Value / AOV) pada kategori-kategori utama tersebut?"*


In [ ]:
# Agregasi data per kategori produk
cat_summary = clean_df.groupby('product_category_name_english').agg(
    total_revenue=('total_order_value', 'sum'),
    total_orders=('order_id', 'nunique'),
    total_items=('product_id', 'count')
).reset_index()

# Hitung AOV dan persentase kumulatif Pareto
cat_summary['aov'] = (cat_summary['total_revenue'] / cat_summary['total_orders']).round(2)
cat_summary = cat_summary.sort_values(by='total_revenue', ascending=False).reset_index(drop=True)
cat_summary['revenue_pct'] = (cat_summary['total_revenue'] / cat_summary['total_revenue'].sum()) * 100
cat_summary['cumulative_pct'] = cat_summary['revenue_pct'].cumsum().round(2)

total_cats = len(cat_summary)
core_80_cats = (cat_summary['cumulative_pct'] <= 80).sum() + 1

print(f"Total kategori produk aktif: {total_cats}")
print(f"Jumlah kategori penyumbang 80% pendapatan kumulatif: {core_80_cats} kategori ({core_80_cats/total_cats:.1%})")
print("\n--- Top 10 Kategori Kontributor Pendapatan Terbesar ---")
print(cat_summary[['product_category_name_english', 'total_revenue', 'total_orders', 'aov', 'revenue_pct', 'cumulative_pct']].head(10))


#### Visualisasi Diagram Pareto Kategori Produk


In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 7))

# Tampilkan Top 25 Kategori untuk visualisasi yang jelas
top_n = 25
df_pareto_plot = cat_summary.head(top_n).copy()

# Warna bar: Kategori 80% (Core) warna biru gelap, sisanya abu-abu
colors_pareto = ['#1E3A8A' if i < core_80_cats else '#94A3B8' for i in range(len(df_pareto_plot))]

# Bar chart pendapatan kategori
bars = ax1.bar(df_pareto_plot['product_category_name_english'], df_pareto_plot['total_revenue'] / 1e3, color=colors_pareto, width=0.65)
ax1.set_ylabel('Pendapatan (Ribu BRL)', fontsize=12, weight='bold', color='#1E3A8A')
ax1.set_title('Diagram Pareto 80/20: Kontribusi Pendapatan Kumulatif Kategori Produk', fontsize=14, weight='bold', pad=15)
plt.xticks(rotation=75, ha='right', fontsize=10)

# Sumbu kedua: Garis persentase kumulatif
ax2 = ax1.twinx()
ax2.plot(df_pareto_plot['product_category_name_english'], df_pareto_plot['cumulative_pct'], color='#DC2626', marker='o', linewidth=2.5, label='Persentase Kumulatif')
ax2.axhline(80, color='#DC2626', linestyle='--', linewidth=1.5, label='Garis Batas Pareto 80%')
ax2.set_ylabel('Persentase Kumulatif (%)', fontsize=12, weight='bold', color='#DC2626')
ax2.set_ylim(0, 105)
ax2.grid(False)

# Legenda
ax2.legend(loc='lower right', frameon=True)
plt.tight_layout()
plt.show()


💡 **Insight Bisnis & Statistik (Pertanyaan 2):**
1. **Validasi Prinsip Pareto (80/20 Rule):** Dari total 72 kategori produk aktif (termasuk `'others'`), tepat **18 kategori (25,0%)** menyumbang **80% total pendapatan** e-commerce. Sisanya sebanyak 54 kategori (75%) merupakan produk *long-tail* dengan kontribusi marjinal.
2. **Kategori Unggulan Utama:** Tiga kategori penyumbang omset tertinggi adalah:
   - `health_beauty`: BRL 1,41 Juta (9,16% pendapatan total, AOV: BRL 163,50)
   - `watches_gifts`: BRL 1,26 Juta (8,20% pendapatan total, AOV: BRL 229,77)
   - `bed_bath_table`: BRL 1,22 Juta (7,97% pendapatan total, AOV: BRL 132,15)
3. **Peluang Monetisasi Berdasarkan AOV:** Kategori `watches_gifts` dan `cool_stuff` (AOV ~BRL 194–230) adalah kategori bernilai tinggi (*high-ticket item*), sedangkan `bed_bath_table` adalah penggerak volume transaksi (*volume driver*). Rekomendasi bisnis: prioritaskan alokasi inventaris, slot promosi, dan promosi *bundling* produk bernilai tinggi pada 18 kategori inti ini.


### 3.3 Pertanyaan 3: Pengaruh Durasi & Keterlambatan Pengiriman terhadap Kepuasan Pelanggan
> *"Bagaimana hubungan korelasi antara durasi pengiriman riil serta keterlambatan pengiriman terhadap skor ulasan pelanggan (review score 1–5), dan seberapa signifikan penurunan skor kepuasan pada pesanan yang terlambat dibandingkan pesanan tepat waktu?"*


In [ ]:
# Agregasi pada tingkat pesanan unik (order-level) untuk menghindari bias pesanan multi-item
unique_orders_reviewed = clean_df.drop_duplicates(subset=['order_id']).dropna(subset=['review_score']).copy()

# Statistik durasi pengiriman (Mean vs Median)
print("--- Statistik Deskriptif Durasi Pengiriman (Hari - Level Pesanan Unik) ---")
deliv_stats = unique_orders_reviewed['actual_delivery_days'].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99])
print(deliv_stats)

print(f"\nMean Durasi Pengiriman: {unique_orders_reviewed['actual_delivery_days'].mean():.2f} hari")
print(f"Median Durasi Pengiriman: {unique_orders_reviewed['actual_delivery_days'].median():.2f} hari")

# Analisis Korelasi (Pearson & Spearman)
pearson_corr = unique_orders_reviewed['actual_delivery_days'].corr(unique_orders_reviewed['review_score'], method='pearson')
spearman_corr = unique_orders_reviewed['actual_delivery_days'].corr(unique_orders_reviewed['review_score'], method='spearman')
delay_corr = unique_orders_reviewed['delay_days'].corr(unique_orders_reviewed['review_score'], method='pearson')

print(f"\nKorelasi Pearson (Waktu Pengiriman vs Review Score): {pearson_corr:.3f}")
print(f"Korelasi Spearman (Waktu Pengiriman vs Review Score): {spearman_corr:.3f}")
print(f"Korelasi Pearson (Hari Keterlambatan vs Review Score): {delay_corr:.3f}")

# Analisis Satisfaction Gap (On-time vs Delayed dengan Normalisasi Kalender)
on_time_stats = unique_orders_reviewed[~unique_orders_reviewed['is_delayed']]['review_score']
late_stats = unique_orders_reviewed[unique_orders_reviewed['is_delayed']]['review_score']

gap = on_time_stats.mean() - late_stats.mean()
print(f"\nRata-rata Skor Pesanan On-Time: {on_time_stats.mean():.2f} (Median: {on_time_stats.median():.1f})")
print(f"Rata-rata Skor Pesanan Late: {late_stats.mean():.2f} (Median: {late_stats.median():.1f})")
print(f"Satisfaction Gap (Penurunan Rating): -{gap:.2f} Bintang")
print(f"Persentase Pesanan Terlambat (Delayed Rate): {unique_orders_reviewed['is_delayed'].mean():.2%}")


#### Visualisasi Pengaruh Durasi Pengiriman & Keterlambatan terhadap Rating


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Subplot 1: Perbandingan Rating On-Time vs Late
categories = ['Tepat Waktu / Lebih Awal\n(On-Time: 89k pesanan)', 'Terlambat dari Estimasi\n(Delayed: 6.5k pesanan)']
means = [on_time_stats.mean(), late_stats.mean()]
colors_gap = ['#10B981', '#EF4444']

bars = ax1.bar(categories, means, color=colors_gap, width=0.45)
ax1.set_ylabel('Rata-rata Skor Ulasan (Skala 1-5 Bintang)', fontsize=12, weight='bold')
ax1.set_ylim(0, 5.2)
ax1.set_title('Satisfaction Gap: Pesanan On-Time vs Terlambat', fontsize=13, weight='bold')

for bar in bars:
    yval = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, yval + 0.1, f"{yval:.2f} ⭐", ha='center', va='bottom', fontsize=12, weight='bold')

ax1.axhline(on_time_stats.mean(), color='#10B981', linestyle='--', alpha=0.5)
ax1.axhline(late_stats.mean(), color='#EF4444', linestyle='--', alpha=0.5)

# Subplot 2: Boxplot Distribusi Waktu Pengiriman per Nilai Rating
sns.boxplot(data=unique_orders_reviewed, x='review_score', y='actual_delivery_days', palette='Blues_r', ax=ax2, showfliers=False)
ax2.set_xlabel('Skor Ulasan Pelanggan (Review Score)', fontsize=12, weight='bold')
ax2.set_ylabel('Durasi Pengiriman Riil (Hari)', fontsize=12, weight='bold')
ax2.set_title('Distribusi Waktu Pengiriman per Tingkat Rating (Tanpa Outlier Ekstrem)', fontsize=13, weight='bold')

plt.tight_layout()
plt.show()


💡 **Insight Bisnis & Statistik (Pertanyaan 3):**
1. **Kemiringan Distribusi (Skewness):** Pada tingkat pesanan unik, rata-rata waktu pengiriman adalah **12,54 hari**, sedangkan median adalah **10,21 hari** (terdapat *right-skewness* akibat pesanan ke wilayah pelosok dengan durasi >100 hari). Penggunaan **Median** lebih tepat merepresentasikan pengalaman tipikal mayoritas pelanggan.
2. **Korelasi Negatif Kuat:** Korelasi Pearson bernilai **-0,335**, mengonfirmasi hipotesis bahwa semakin lama barang tiba, semakin rendah kepuasan pelanggan.
3. **Penurunan Rating yang Sangat Tajam (Satisfaction Gap):**
   - Pesanan **On-Time** memperoleh rata-rata rating **4,29 bintang** (Median: 5.0 bintang).
   - Pesanan **Terlambat (Delayed)** anjlok menjadi rata-rata **2,27 bintang** (Median: 2.0 bintang).
   - Terjadi **Satisfaction Gap drastis sebesar -2,02 bintang** ketika pesanan melewati batas tanggal estimasi kalender! Hal ini membuktikan bahwa batas tanggal estimasi adalah batas toleransi psikologis utama bagi pelanggan.


## 🎯 4. Kesimpulan & Rekomendasi Bisnis Strategis

Berdasarkan analisis empiris ketiga pilar bisnis di atas, berikut adalah rekomendasi komersial dan operasional yang dapat ditindaklanjuti:

### 1. Strategi Pertumbuhan Penjualan (Commercial Growth)
- **Kapitalisasi Momen Musiman:** Lonjakan Black Friday November 2017 (+53,5% MoM) membuktikan tingginya sensitivitas konsumen terhadap promo akhir tahun. Tim komersial harus menyiapkan kampanye *pre-sale* sejak Oktober dan menjaga stok pada kategori unggulan.
- **Fokus Akuisisi Pelanggan:** Mengingat AOV stabil di kisaran ~BRL 160, pertumbuhan pendapatan paling efektif digerakkan oleh peningkatan volume pesanan baru dan promosi *cross-selling*.

### 2. Efisiensi Portofolio Produk (Merchandising & Inventory)
- **Penerapan Alokasi Pareto 80/20:** Alokasikan 80% anggaran promosi digital dan ruang etalase utama hanya pada **18 kategori inti** (khususnya `health_beauty`, `watches_gifts`, dan `bed_bath_table`).
- **Strategi Bundling:** Kombinasikan kategori bervolume tinggi (`bed_bath_table`) dengan produk bernilai tinggi (`watches_gifts` / `cool_stuff`) untuk mengerek Average Order Value.

### 3. Keunggulan Operasional & Logistik (Customer Retention & SLA)
- **Perlindungan Estimasi Pengiriman (SLA Safeguard):** Karena pesanan terlambat mengalami penurunan skor hingga -2,02 bintang, sistem harus memberikan estimasi tanggal pengiriman yang realistis dengan *buffer* 2–3 hari pada rute rawan macet.
- **Proactive Early Warning & Compensatory Voucher:** Jika terdeteksi pesanan akan terlambat, kirimkan notifikasi proaktif beserta voucher kompensasi belanja berikutnya sebelum pelanggan memberikan ulasan bintang 1.
